# Variant-effect prediction and structure: multipass membrane proteins vs all other proteins

This notebook asks whether multipass membrane proteins are a blind spot for current variant-effect prediction, which would motivate a systematic mutagenesis study of membrane proteins. It is organised around three questions:

1. **Clinical performance.** Do predictors separate ClinVar pathogenic from benign variants less well in membrane proteins?
2. **Dynamic range and error.** Are scores compressed, mis-calibrated, or more contradictory in membrane proteins, and where do variants of uncertain significance accumulate?
3. **Structure.** How do membrane proteins differ in shape, packing and the distribution of hydrophobicity, in ways that predictors trained mostly on soluble proteins would not capture?

**Groups.** Every protein has one class from UniProt annotation (`class_annotation` from the shape pipeline). **Multipass membrane** (orange) is the group of interest; single-pass, beta-barrel, soluble and ambiguous (peripheral, lipid-anchored or conflicting annotation) proteins are each shown, plus all non-multipass proteins pooled.

**Conventions.** Every predictor is oriented so that a higher score means more likely pathogenic, so an AUC above 0.5 is always better than chance. Confidence intervals resample whole proteins, because variants of one protein are not independent. Multiple comparisons are BH-adjusted across predictors. Clinical analyses use human proteins only (yeast has no ClinVar); structural analyses use both organisms.

In [ ]:
import os, sys, warnings
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import Markdown, display
sys.path.insert(0, str(Path.cwd().parent))          # membrane_analysis/ (when not pip-installed)
warnings.filterwarnings("ignore", category=RuntimeWarning)
from mvsa import analyses as A, plots as P, report as R, stats as S
from mvsa.data import load, available_predictors
from mvsa.groups import CLASS_ORDER, CLASS_LABEL, CLASS_COLOR, MEMBRANE, MAIN_PREDICTORS, PREDICTOR_GROUP, DIRECTION

# ---------------------------------------------------------------- parameters
DATA_DIR = Path(os.environ.get("MVSA_DATA", "../data"))     # written by `python -m mvsa.prepare`
FIG_DIR = Path(os.environ.get("MVSA_FIGURES", "../figures"))
INCLUDE_LIKELY = True       # count likely pathogenic / likely benign as labels
MIN_STARS = 0               # ClinVar review filter: 0 all, 1 drop 'no assertion criteria', 2 multiple submitters+
N_BOOT = int(os.environ.get("MVSA_NBOOT", 300))   # protein bootstrap resamples for CIs
MIN_PER_CLASS = 5           # per-protein AUCs need this many pathogenic AND benign variants
HUMAN = 9606

P.setup(FIG_DIR)
D = load(DATA_DIR)
prot = D["proteins"]
lab_all = D["labelled_variants"]
lab = S.labels(lab_all, include_likely=INCLUDE_LIKELY, min_stars=MIN_STARS)     # pathogenic (y=1) / benign (y=0)
lab = lab[lab.protein_class.notna()]
sample = D["variant_sample"]
PRED = [p for p in MAIN_PREDICTORS if p in available_predictors(lab)]
VEP = [p for p in ("alphamissense", "esm1v", "saprot") if p in PRED]
print("data:", D["meta"].get("created", "?"), "| predictors with data:", ", ".join(PRED))
missing = [p for p in MAIN_PREDICTORS if p not in PRED]
if missing:
    print("no data yet (skipped):", ", ".join(missing))

## 0. Data overview

In [ ]:
ov = A.overview(prot[prot.taxid == HUMAN], lab_all)
ov

In [ ]:
cov = pd.DataFrame({c: [lab.loc[lab.protein_class == c, p].notna().mean() for p in PRED] for c in CLASS_ORDER}, index=PRED)
print('Fraction of labelled variants with a score, by class:')
cov.round(2)

## 1. Clinical performance: pathogenic vs benign

### 1.1 Pooled across proteins
AUC over every labelled variant of each class, with protein-bootstrap 95% CIs. The right panel shows multipass minus each comparator; values below zero mean predictors do worse on multipass proteins.

In [ ]:
pooled = A.pooled_performance(lab, PRED, n_boot=N_BOOT)
fig, ax = plt.subplots(1, 2, figsize=(12, 0.45 * len(PRED) + 1.6), gridspec_kw={'width_ratios': [1.2, 1]})
P.forest(pooled[pooled['class'].isin(CLASS_ORDER)], 'predictor', 'class', ax=ax[0], groups=CLASS_ORDER,
         title='AUC by protein class')
d = pooled[pooled['class'].isin(['SOLUBLE', 'NON_MULTIPASS'])].rename(columns={'mp_minus_delta': 'dd', 'mp_minus_delta_low': 'dl', 'mp_minus_delta_high': 'dh'})
P.forest(d, 'predictor', 'class', value='dd', lo='dl', hi='dh', ref=0, ax=ax[1], xlabel='AUC difference (multipass - comparator)',
         title='Multipass minus comparator')
ax[1].set_yticklabels([])
fig.tight_layout(); P.save(fig, '1_1_pooled_auc')

In [ ]:
cols = ['predictor', 'class', 'n_proteins', 'n_pathogenic', 'n_benign', 'auc', 'ci_low', 'ci_high', 'mp_minus_delta', 'mp_minus_delta_low', 'mp_minus_delta_high', 'mp_minus_q']
pooled[cols].round(3)

### 1.2 ROC curves

In [ ]:
show = [p for p in ('alphamissense', 'esm1v', 'thermompnn', 'blosum62') if p in PRED]
fig, axes = plt.subplots(1, len(show), figsize=(3.6 * len(show), 3.4), squeeze=False)
for ax, p in zip(axes[0], show):
    for c, (fpr, tpr, a) in A.roc_curves(lab, p, ['MULTIPASS', 'SOLUBLE', 'SINGLE_PASS', 'AMBIGUOUS']).items():
        ax.plot(fpr, tpr, color=CLASS_COLOR[c], lw=2 if c == MEMBRANE else 1.3, label=f'{CLASS_LABEL[c]} {a:.2f}')
    ax.plot([0, 1], [0, 1], color=P.AXIS, lw=0.8); ax.set_title(P.label(p), loc='left')
    ax.set_xlabel('False positive rate'); ax.set_ylabel('True positive rate'); ax.legend(fontsize=7, loc='lower right')
fig.tight_layout(); P.save(fig, '1_2_roc')

### 1.3 Per protein
AUC inside each protein (from the ve_simple tables), for proteins with at least `MIN_PER_CLASS` pathogenic and benign variants. This removes any between-gene component but keeps fewer proteins.

In [ ]:
pp, pp_comp = A.per_protein_performance(D['protein_auc'], PRED, MIN_PER_CLASS)
show = [p for p in PRED if p in set(pp.predictor)][:6]
fig, axes = plt.subplots(1, len(show), figsize=(3.3 * len(show), 3.4), squeeze=False)
for ax, p in zip(axes[0], show):
    P.class_violin(pp[pp.predictor == p], 'auc', [c for c in CLASS_ORDER if c != 'BETA_BARREL_MEMBRANE'], ax=ax, ylabel='per-protein AUC', title=P.label(p), ref=0.5)
fig.tight_layout(); P.save(fig, '1_3_per_protein_auc')
pp_comp.round(3)

### 1.4 Where in the protein
Inside multipass proteins, variants are split by location: transmembrane segment facing the lipid (exposed in the AlphaFold model) or buried, juxtamembrane (within 5 residues of a TM segment), and extramembrane buried or exposed. Soluble proteins are split into buried and exposed residues for reference (relative SASA threshold 0.2).

In [ ]:
loc = A.location_performance(lab, PRED, n_boot=max(100, N_BOOT // 2))
loc['group'] = loc['class'].map({'MULTIPASS': 'Multipass', 'SOLUBLE': 'Soluble'}) + ': ' + loc['location']
show = [p for p in ('alphamissense', 'esm1v', 'saprot', 'thermompnn', 'blosum62') if p in PRED]
fig, axes = plt.subplots(1, len(show), figsize=(3.4 * len(show), 3.6), sharey=True, squeeze=False)
for ax, p in zip(axes[0], show):
    d = loc[loc.predictor == p].reset_index(drop=True)
    y = np.arange(len(d))
    for c in d['class'].unique():
        m = (d['class'] == c).to_numpy()
        ax.errorbar(d.auc[m], y[m], xerr=[(d.auc - d.ci_low)[m], (d.ci_high - d.auc)[m]], fmt='o', ms=6, elinewidth=1.4,
                    color=CLASS_COLOR[c], ecolor=CLASS_COLOR[c], mec='white')
    ax.axvline(0.5, color=P.AXIS, lw=1); ax.set_title(P.label(p), loc='left'); ax.set_xlabel('AUC')
    ax.set_yticks(y, [f"{g} (n={a}/{b})" for g, a, b in zip(d.group, d.n_pathogenic, d.n_benign)], fontsize=7)
axes[0][0].invert_yaxis(); fig.tight_layout(); P.save(fig, '1_4_auc_by_location')
loc[['predictor', 'class', 'location', 'n_proteins', 'n_pathogenic', 'n_benign', 'auc', 'ci_low', 'ci_high']].round(3)

### 1.5 Is the gap explained by easier label mixes or structural context?
*Matched* recomputes the AUC within strata of residue pLDDT, burial and the protein's fraction of pathogenic variants, weighting both classes by the multipass strata. *Within-protein* averages AUCs computed inside each protein. If the multipass deficit persists after both, it is not an artefact of which genes or residues happen to be labelled.

In [ ]:
matched = A.matched_performance(lab, [p for p in PRED if p in ('alphamissense', 'esm1v', 'saprot', 'thermompnn', 'foldx', 'blosum62')], n_boot=max(50, N_BOOT // 3))
d = matched.copy(); d['group'] = d.estimate
fig, ax = plt.subplots(figsize=(6.5, 0.5 * d.predictor.nunique() + 1.5))
for j, (k, col) in enumerate(zip(['raw', 'matched', 'within-protein'], ['#8f8d85', '#eb6834', '#6250d6'])):
    s = d[d.estimate == k].reset_index(drop=True); y = np.arange(len(s)) + (j - 1) * 0.22
    ax.errorbar(s.delta, y, xerr=[s.delta - s.delta_low, s.delta_high - s.delta], fmt='o', color=col, label=k, ms=5)
ax.axvline(0, color=P.AXIS); ax.set_yticks(range(s.shape[0]), [P.label(p) for p in s.predictor]); ax.invert_yaxis()
ax.set_xlabel('AUC difference, multipass - soluble'); ax.legend(fontsize=8); fig.tight_layout(); P.save(fig, '1_5_matched')
display(Markdown(R.matched_takeaway(matched)))

### Takeaway (i)

In [ ]:
display(Markdown(R.performance_takeaway(pooled, 'SOLUBLE')))
display(Markdown(R.performance_takeaway(pooled, 'NON_MULTIPASS')))

## 2. Dynamic range and error

### 2.1 Score distributions over all possible substitutions
Every possible missense change (a random sample of the proteome-wide saturation tables), oriented so higher = more pathogenic.

In [ ]:
show = [p for p in PRED if p in sample][:8]
fig, axes = plt.subplots(2, (len(show) + 1) // 2, figsize=(3.5 * ((len(show) + 1) // 2), 6.4), squeeze=False)
for ax, p in zip(axes.ravel(), show):
    P.ecdf(sample.assign(v=sample[p] * DIRECTION[p]), 'v', ['MULTIPASS', 'SINGLE_PASS', 'SOLUBLE', 'AMBIGUOUS'], ax=ax, xlabel=P.label(p) + ' (oriented)')
for ax in axes.ravel()[len(show):]: ax.axis('off')
fig.tight_layout(); P.save(fig, '2_1_score_ecdf')

Inside multipass proteins, scores of substitutions in TM segments vs elsewhere:

In [ ]:
mp = sample[sample.protein_class == MEMBRANE]
fig, axes = plt.subplots(1, len(show), figsize=(3 * len(show), 3), squeeze=False)
for ax, p in zip(axes[0], show):
    for region, col in (('TM', '#eb6834'), ('Juxtamembrane', '#eda100'), ('Non-TM', '#2a78d6')):
        v = np.sort((mp.loc[mp.region == region, p] * DIRECTION[p]).dropna())
        if len(v): ax.plot(v, np.arange(1, len(v) + 1) / len(v), color=col, lw=2, label=region)
    ax.set_title(P.label(p), loc='left', fontsize=9); ax.legend(fontsize=7)
fig.tight_layout(); P.save(fig, '2_1_tm_vs_rest_ecdf')
A.tm_vs_rest_scores(D['protein_scores'], PRED).round(3)

### 2.2 Dynamic range per protein
The spread of each protein's scores (5th to 95th percentile over all substitutions), split into between-position spread (how much positions differ) and within-position spread (how much the substitutions at one position differ). A compressed range means the predictor cannot rank variants within that protein.

In [ ]:
dyn = A.dynamic_range(D['protein_scores'], PRED)
ps = D['protein_scores']; ps = ps[ps.region == 'all']
show = [p for p in ('alphamissense', 'esm1v', 'saprot', 'thermompnn', 'blosum62') if p in PRED]
fig, axes = plt.subplots(1, len(show), figsize=(3.3 * len(show), 3.3), squeeze=False)
for ax, p in zip(axes[0], show):
    P.class_violin(ps[ps.predictor == p], 'dynamic_range', [c for c in CLASS_ORDER if c != 'BETA_BARREL_MEMBRANE'], ax=ax, ylabel='5-95% range', title=P.label(p))
fig.tight_layout(); P.save(fig, '2_2_dynamic_range')
dyn.round(3)

### 2.3 Thresholds and error rates
A threshold tuned on soluble proteins (Youden's J), or AlphaMissense's published cut-off, applied unchanged to every class. A higher false-negative rate in multipass proteins means pathogenic variants are missed there.

In [ ]:
thr = A.threshold_transfer(lab, PRED)
d = thr[thr.threshold == 'Youden on soluble']
fig, axes = plt.subplots(1, 2, figsize=(12, 0.42 * d.predictor.nunique() + 1.6))
for ax, m, t in zip(axes, ['fnr', 'fpr'], ['False-negative rate', 'False-positive rate']):
    w = d.pivot_table(index='predictor', columns='class', values=m).reindex(columns=[c for c in CLASS_ORDER if c in set(d['class'])])
    h = 0.8 / len(w.columns)
    for j, c in enumerate(w.columns):
        ax.barh(np.arange(len(w)) + j * h, w[c], height=h, color=CLASS_COLOR[c], label=CLASS_LABEL[c])
    ax.set_yticks(np.arange(len(w)) + 0.4 - h / 2, [P.label(p) for p in w.index]); ax.invert_yaxis(); ax.set_xlabel(t)
axes[0].legend(fontsize=7); fig.tight_layout(); P.save(fig, '2_3_error_rates')
display(Markdown(R.threshold_takeaway(thr)))
am = A.alphamissense_zones(lab_all[lab_all.protein_class.notna()])
am.round(3)

### 2.4 Calibration
Each predictor is mapped to a probability of pathogenicity with a logistic fit on soluble variants. ClinVar's pathogenic fraction differs between classes for reasons of ascertainment, so probabilities are shifted to each class's own pathogenic fraction before checking them. If the map then transfers, multipass variants lie on the diagonal; remaining deviations mean the same score carries different information in membrane proteins.

In [ ]:
calib, curves = A.calibration(lab, PRED)
show = [p for p in ('alphamissense', 'esm1v', 'saprot', 'thermompnn') if p in set(calib.predictor)]
fig, axes = plt.subplots(1, max(1, len(show)), figsize=(3.4 * max(1, len(show)), 3.4), squeeze=False)
for ax, p in zip(axes[0], show):
    for c in ('SOLUBLE', 'MULTIPASS', 'SINGLE_PASS'):
        r = curves[(curves.predictor == p) & (curves['class'] == c)]
        if len(r): ax.plot(r.pred, r.obs, 'o-', color=CLASS_COLOR[c], ms=4, lw=1.6, label=CLASS_LABEL[c])
    ax.plot([0, 1], [0, 1], color=P.AXIS, lw=0.8); ax.set_title(P.label(p), loc='left')
    ax.set_xlabel('Predicted P(pathogenic), prevalence-adjusted'); ax.set_ylabel('Observed fraction pathogenic'); ax.legend(fontsize=7)
fig.tight_layout(); P.save(fig, '2_4_calibration')
display(Markdown(R.calibration_takeaway(calib)))
calib.round(3)

### 2.5 Do predictors agree?
Rank correlation between predictors over all substitutions, per class, and the per-variant spread of the predictors' proteome percentiles (higher = more disagreement).

In [ ]:
corr = A.correlation_by_class(sample, PRED)
if MEMBRANE in corr and 'SOLUBLE' in corr:
    fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
    P.heatmap(corr[MEMBRANE], ax=axes[0], title='Multipass: Spearman')
    P.heatmap(corr['SOLUBLE'], ax=axes[1], title='Soluble: Spearman')
    P.heatmap(corr[MEMBRANE] - corr['SOLUBLE'], ax=axes[2], title='Difference (multipass - soluble)', vmin=-0.5, vmax=0.5)
    fig.tight_layout(); P.save(fig, '2_5_predictor_correlation')

In [ ]:
dis_lab = A.disagreement(lab, sample, [p for p in PRED if p != 'solvent_exposure'])
dis_all = A.disagreement(sample, sample, [p for p in PRED if p != 'solvent_exposure'])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
P.class_violin(dis_all, 'disagreement_sd', [c for c in CLASS_ORDER if c != 'BETA_BARREL_MEMBRANE'], ax=axes[0], ylabel='SD of predictor percentiles', title='All substitutions')
d = dis_all[dis_all.protein_class == MEMBRANE]
P.class_violin(d.assign(g=d.region), 'disagreement_sd', ['TM', 'Juxtamembrane', 'Non-TM'], group='g', ax=axes[1], ylabel='SD of predictor percentiles', title='Multipass, by region')
fig.tight_layout(); P.save(fig, '2_5_disagreement')
dis_all.groupby('protein_class').disagreement_sd.describe().round(2)

### 2.6 Variants of uncertain significance
Where VUS accumulate, and how predictors would call them using thresholds tuned on soluble proteins. A large VUS burden with split predictor calls marks where experimental maps would add most.

In [ ]:
vb = A.vus_burden(prot[prot.taxid == HUMAN])
display(vb.round(3))
per_vus, agree = A.vus_reclassification(lab_all[lab_all.protein_class.notna()], PRED, VEP)
if len(agree):
    a = agree.set_index('class').reindex([c for c in CLASS_ORDER + ['NON_MULTIPASS'] if c in set(agree['class'])])
    fig, ax = plt.subplots(figsize=(7, 2.8))
    left = np.zeros(len(a))
    for k, col in (('all pathogenic', '#e34948'), ('split', '#eda100'), ('all benign', '#2a78d6')):
        ax.barh(range(len(a)), a[k], left=left, color=col, label=k); left += a[k].fillna(0).to_numpy()
    ax.set_yticks(range(len(a)), [CLASS_LABEL[c] for c in a.index]); ax.invert_yaxis(); ax.set_xlabel('Fraction of VUS')
    ax.set_title('Agreement of ' + ', '.join(P.label(v) for v in VEP) + ' on VUS', loc='left'); ax.legend(fontsize=7, ncol=3, loc='lower right')
    fig.tight_layout(); P.save(fig, '2_6_vus_agreement')
per_vus.pivot_table(index='predictor', columns='class', values='called_pathogenic').round(3)

## 3. Structural differences

### 3.1 Model confidence and disorder
All proteins (human and yeast). `frac_excluded` is the fraction of residues removed as low-confidence or outside any TED domain.

In [ ]:
pr = prot.rename(columns={'class_annotation': 'protein_class'})
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4))
P.class_violin(pr, 'plddt_mean', CLASS_ORDER, ax=axes[0], ylabel='mean pLDDT')
P.class_violin(pr, 'frac_excluded', CLASS_ORDER, ax=axes[1], ylabel='fraction excluded')
fig.tight_layout(); P.save(fig, '3_1_confidence')

### 3.2 Shape, compactness and hydrophobicity: size-matched effects
Cliff's delta of multipass vs each class within size bins (so larger membrane proteins are only compared with proteins of similar size); +1 means always higher in multipass.

In [ ]:
sh = D['shape_units']
con_t = A.structural_contrasts(sh, 'CHAIN_TRIMMED')
con_d = A.structural_contrasts(sh, 'DOMAIN')
fig, axes = plt.subplots(1, 2, figsize=(12, 0.42 * con_t.metric.nunique() + 1.6), sharey=True)
for ax, con, t in zip(axes, [con_t, con_d], ['Trimmed chains', 'TED domains']):
    d = con[con['class'].isin(['SOLUBLE', 'SINGLE_PASS', 'NON_MULTIPASS'])].assign(lo=lambda x: x.cliffs_delta_size_matched, hi=lambda x: x.cliffs_delta_size_matched)
    P.forest(d, 'metric', 'class', value='cliffs_delta_size_matched', lo='lo', hi='hi', ref=0, ax=ax, xlabel="Cliff's delta (multipass vs class)", title=t)
fig.tight_layout(); P.save(fig, '3_2_structure_effects')
display(Markdown(R.structure_takeaway(con_t)))

### 3.3 Burial and environment of residues by location

In [ ]:
res = D['residues']
bl = A.burial_by_location(res)
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
order = [('MULTIPASS', l) for l in ['TM lipid-facing', 'TM buried', 'Juxtamembrane', 'Extramembrane exposed', 'Extramembrane buried']] + [('SOLUBLE', 'Exposed'), ('SOLUBLE', 'Buried')]
for ax, m, t in zip(axes, ['median_depth', 'frac_helix'], ['Median burial depth (A)', 'Fraction helix']):
    v = [bl[(bl.protein_class == c) & (bl.location == l)][m].squeeze() if len(bl[(bl.protein_class == c) & (bl.location == l)]) else np.nan for c, l in order]
    ax.barh(range(len(order)), v, color=[CLASS_COLOR[c] for c, _ in order]); ax.set_yticks(range(len(order)), [f'{c.title()}: {l}' for c, l in order], fontsize=8)
    ax.invert_yaxis(); ax.set_xlabel(t)
fig.tight_layout(); P.save(fig, '3_3_burial_by_location')
bl.round(3)

### 3.4 Structure and prediction quality
Does predictive performance depend on structural context differently in membrane proteins? Pooled AUC within bins of residue burial depth and model confidence, and per-protein AUC vs protein-level structure.

In [ ]:
p0 = VEP[0] if VEP else PRED[0]
ctx_depth = A.auc_by_context(lab, p0, 'depth_mean', [-0.1, 1, 3, 6, 100], ['surface (<1 A)', '1-3 A', '3-6 A', 'core (>6 A)'], n_boot=max(50, N_BOOT // 3))
ctx_plddt = A.auc_by_context(lab, p0, 'plddt_model', [0, 70, 90, 101], ['<70', '70-90', '>=90'], n_boot=max(50, N_BOOT // 3))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.4))
for ax, d, t in zip(axes, [ctx_depth, ctx_plddt], ['by burial depth', 'by pLDDT']):
    for j, c in enumerate([MEMBRANE, 'SOLUBLE']):
        s = d[d['class'] == c].reset_index(drop=True); x = np.arange(len(s)) + (j - 0.5) * 0.25
        ax.errorbar(x, s.auc, yerr=[s.auc - s.ci_low, s.ci_high - s.auc], fmt='o', color=CLASS_COLOR[c], label=CLASS_LABEL[c])
    ax.set_xticks(range(len(s)), s['bin']); ax.axhline(0.5, color=P.AXIS); ax.set_ylabel(P.label(p0) + ' AUC'); ax.set_title(t, loc='left'); ax.legend(fontsize=7)
fig.tight_layout(); P.save(fig, '3_4_auc_by_context')
A.auc_vs_structure(D['protein_auc'], sh, p0, A.STRUCT_METRICS, MIN_PER_CLASS).round(3)

## 4. Summary: multipass vs soluble across all analyses

In [ ]:
summary = R.summary_table(pooled, matched, thr, calib, dyn, con_t)
summary.to_csv(FIG_DIR / 'summary_multipass_vs_soluble.tsv', sep='\t', index=False)
print('Figures and summary table written to', FIG_DIR.resolve())
summary.round(3)